In [4]:
from src.app_utils import get_player_features, get_all_players_for_round, get_db_connection
import pandas as pd

In [5]:
from sqlalchemy import create_engine
import pandas as pd
from src.utils import add_matchup_probabilities, create_advanced_features, get_team_stats
import numpy as np
import pandas as pd

##  Sample from postgre

In [6]:
username = 'rodrigo'
host = 'localhost'           
port = '5432'               
database = 'futmondo_full_players_info'

In [7]:
engine = create_engine(f'postgresql+psycopg2://{username}@{host}:{port}/{database}')

In [8]:
query = "SELECT * FROM player_points_def"
df = pd.read_sql(query, engine)
df.shape

(7363, 17)

In [9]:
df_yamal=df[df['name']=='Lamine Yamal']

In [10]:
df_yamal.columns

Index(['player_id', 'name', 'role', 'round', 'team_id', 'home_average',
       'away_average', 'overall_average', 'current_price', 'matches_played',
       'rating', 'match_minus_1', 'match_minus_2', 'last_2_average',
       'target_points', 'unique_id', 'team'],
      dtype='object')

In [11]:
df_yamal[['name', 'round', 'home_average', 'away_average', 'current_price', 'matches_played', 'rating', 'match_minus_1', 'match_minus_2', 'target_points']]

,name,round,home_average,away_average,current_price,matches_played,rating,match_minus_1,match_minus_2,target_points
328,Lamine Yamal,13,8.333333,8.333333,131081702,12,7,9.0,10,11.0
843,Lamine Yamal,14,8.333333,8.333333,131081702,12,7,11.0,9,7.0
1359,Lamine Yamal,15,8.333333,8.333333,131081702,12,7,7.0,11,9.0
1873,Lamine Yamal,16,8.333333,8.333333,131081702,12,7,9.0,7,9.0
2415,Lamine Yamal,17,8.142857,8.375000,143225867,15,7,9.0,12,7.0
2927,Lamine Yamal,18,8.142857,8.777778,147049863,16,7,7.0,9,12.0
3448,Lamine Yamal,19,8.142857,8.777778,147049863,16,7,12.0,7,5.0
3961,Lamine Yamal,20,8.142857,8.777778,147049863,16,7,5.0,12,12.0
4484,Lamine Yamal,21,9.222222,9.300000,161036274,19,7,12.0,5,14.0
5014,Lamine Yamal,22,9.222222,9.300000,161036274,19,7,14.0,12,14.0


In [12]:
# # make sure data is sorted correctly
# df = df.sort_values(by=["player_id", "round"])

# # shift match_minus_1 one round backwards per player
# df["next_match_minus_1"] = (
#     df.groupby("player_id")["match_minus_1"]
#       .shift(-1)
# )

# # assign only for round 17
# df.loc[df["round"] == 17, "target_points"] = (
#     df.loc[df["round"] == 17, "next_match_minus_1"]
# )

# # optional: clean up helper column
# df.drop(columns="next_match_minus_1", inplace=True)


In [13]:
# round 17-21 is correct now

In [14]:
# df_tol=df[df['name']=='Mikautadze'].sort_values(by='round', ascending=False)

# df_tol[['match_minus_1', 'match_minus_2', 'last_2_average',
#        'target_points', 'round']]

### Get all data loading in one

In [15]:
import requests
import json
import pandas as pd
import numpy as np
import logging
from sqlalchemy import create_engine
from src.utils import (
    create_round_features,
    add_matchup_probabilities,
    create_advanced_features,
    get_team_stats,
    standardize_team_names,
    predict_upcoming_matches,
    TEAM_ID_MAPPING,
    TEAM_NAME_MAPPING
)

In [16]:
def fetch_championship_players(auth_token, user_id, championship_id):
    """Fetch all players from championship endpoint"""
    headers = {
        'Authorization': f'Bearer {auth_token}',
        'x-futmondo-token': auth_token,
        'x-futmondo-userid': user_id
    }
    
    endpoint_url = 'https://api.futmondo.com/5/league/championshipplayers'
    query_params = {'championshipId': championship_id}
    
    logging.info(f"Fetching all players from: {endpoint_url}")
    
    try:
        response = requests.post(endpoint_url, headers=headers, json={'query': query_params})
        
        if response.status_code == 200:
            full_response = response.json()
            player_list = full_response.get('answer', {}).get('players')
            
            if isinstance(player_list, list):
                logging.info(f"Successfully extracted {len(player_list)} player records")
                return player_list
            else:
                logging.error("Player list not found in expected structure")
                return None
        else:
            logging.error(f"Failed to fetch players: {response.status_code}")
            return None
            
    except requests.exceptions.RequestException as e:
        logging.error(f"Request error: {e}")
        return None


def clean_numpy_values(df):
    """Convert numpy types to native Python types"""
    def clean_value(x):
        if isinstance(x, (np.floating, np.integer)):
            return x.item()
        if isinstance(x, np.bool_):
            return bool(x)
        return x
    
    return df.applymap(clean_value)


In [17]:
# Configuration
AUTH_TOKEN = '5e65_20f468a19ad19ef73979642df99d6603'
USER_ID = '56c6b62085617f9b1dc7d061'
CHAMPIONSHIP_ID = '5f7b19924dcd043e8a092dd4'

db_config = {
    'username': 'rodrigo',
    'host': 'localhost',
    'port': '5432',
    'database': 'futmondo_full_players_info'
}

# Fetch and save player data
player_list = fetch_championship_players(AUTH_TOKEN, USER_ID, CHAMPIONSHIP_ID)
if not player_list:
    logging.error("Failed to fetch player data. Aborting.")

last-round-23

In [18]:
# df_cabrera=df[df['name']=='Cabrera'].sort_values(by='round', ascending=False)

In [19]:
# output_path = '/Users/rodrigo/football-data-analytics/futmondo_points_predict/data/detailed_scrapped/all_detailed_players_def.json'

# # Create directory if it doesn't exist
# import os
# os.makedirs(os.path.dirname(output_path), exist_ok=True)

# with open(output_path, 'w', encoding='utf-8') as f:
#     json.dump(player_list, f, indent=2, ensure_ascii=False)
# logging.info(f"Saved player data to {output_path}")

# Create rolling features
df_rolling = create_round_features(players_list=player_list, target_rounds=[23, 24, 25, 26])


/Users/rodrigo/football-data-analytics/futmondo_points_predict/src/utils.py:475: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(all_rounds, ignore_index=True)


In [20]:
df_rolling[df_rolling['name']=='Lamine Yamal'].sort_values(by='round', ascending=False)

,player_id,name,role,round,team_id,home_average,away_average,overall_average,current_price,matches_played,rating,match_minus_1,match_minus_2,last_2_average,target_points,unique_id
1822,644d90d4479a44042784eb9f,Lamine Yamal,delantero,26,504e581e4d8bec9a670000c7,8.9,8.363636,8.619048,161975535,21,7,6,-1,2.5,NaN,651df1dff304b0ed70bd101f4ddf7f76e4e6dfbdb1ac11...
1286,644d90d4479a44042784eb9f,Lamine Yamal,delantero,25,504e581e4d8bec9a670000c7,8.9,8.363636,8.619048,161975535,21,7,-1,12,5.5,6.0,ca81fce710cb758bde27b9d9051476a4ca3a96725ab654...
750,644d90d4479a44042784eb9f,Lamine Yamal,delantero,24,504e581e4d8bec9a670000c7,8.9,8.363636,8.619048,161975535,21,7,12,14,13.0,-1.0,c0bc665aa4df02fe783d8c99897ca3ac587fbfef0870c2...
214,644d90d4479a44042784eb9f,Lamine Yamal,delantero,23,504e581e4d8bec9a670000c7,8.9,8.363636,8.619048,161975535,21,7,14,14,14.0,12.0,f0807e6d6a69c674558b8f92def33d21ad22b8bf2628e2...


In [21]:
# Remove specific team IDs
ids_to_remove = ['51ffb6b7113981890700003a', '5211d81592d57d145a0000ce', '520e4ee4a776cc826b00004b']
df_rolling = df_rolling[~df_rolling['team_id'].isin(ids_to_remove)]

# Map team IDs to team names
df_rolling['team'] = df_rolling['team_id'].map(TEAM_ID_MAPPING)

In [22]:
# df[df['name']=='Cabrera'].sort_values(by='round', ascending=False).head(10)

In [23]:
try:
    # existing_df = pd.read_sql('SELECT * FROM player_points', engine)
    final_df = pd.concat([df, df_rolling], ignore_index=True)
    # Remove duplicates keeping the last occurrence (newest data)
    final_df = final_df.drop_duplicates(subset=['round', 'name'], keep='last')
except:
    final_df = df_rolling

In [24]:
# final_df = final_df.sort_values(['player_id', 'round'])
final_df['target_points'] = final_df.groupby('player_id')['match_minus_1'].shift(-1)

In [25]:
final_df[final_df['name']=='Mikautadze'].sort_values(by='round', ascending=False)

,player_id,name,role,round,team_id,home_average,away_average,overall_average,current_price,matches_played,rating,match_minus_1,match_minus_2,last_2_average,target_points,unique_id,team
9421,68b5d6ae026ac303d72e0129,Mikautadze,delantero,26,51b890f5b986415a2c000012,6.500000,6.000000,6.250000,36545571,20,5,3.000000,12,7.5,NaN,0fbb5f2c28ea4fd6a7dc79217f97e3bbe37b19782ce6dc...,Villarreal
8888,68b5d6ae026ac303d72e0129,Mikautadze,delantero,25,51b890f5b986415a2c000012,6.500000,6.000000,6.250000,36545571,20,5,12.000000,9,10.5,3.000000,df4eac71b573841877c4b2255dc8f0d85bd290ac76d3d3...,Villarreal
8355,68b5d6ae026ac303d72e0129,Mikautadze,delantero,24,51b890f5b986415a2c000012,6.500000,6.000000,6.250000,36545571,20,5,9.000000,15,12.0,12.000000,76bb0960e977f36d697168f7f1c4ec5bd0fa6e0e0573d7...,Villarreal
7822,68b5d6ae026ac303d72e0129,Mikautadze,delantero,23,51b890f5b986415a2c000012,6.500000,6.000000,6.250000,36545571,20,5,15.000000,3,9.0,9.000000,7bccd401ed938fb267aba51af25e47672b15ac9c387be0...,Villarreal
5159,68b5d6ae026ac303d72e0129,Mikautadze,delantero,22,51b890f5b986415a2c000012,6.888889,4.875000,5.941176,30083060,17,5,3.000000,3,3.0,15.000000,4e02ab1ee6a58c0fc7a4408ea5c873ba19d88bec171f0a...,Villarreal
4630,68b5d6ae026ac303d72e0129,Mikautadze,delantero,21,51b890f5b986415a2c000012,6.888889,4.875000,5.941176,30083060,17,5,3.000000,10,6.5,3.000000,185ce08c20ba3e208b0c5d0f451b11cd8d10c55d85d398...,Villarreal
4106,68b5d6ae026ac303d72e0129,Mikautadze,delantero,20,51b890f5b986415a2c000012,6.285714,5.142857,5.714286,29203734,14,5,10.000000,9,9.5,3.000000,68c2ea22d4edbb62f4f5d613baf521c8e5c8f428b42c29...,Villarreal
3598,68b5d6ae026ac303d72e0129,Mikautadze,delantero,19,51b890f5b986415a2c000012,6.285714,5.142857,5.714286,29203734,14,5,9.000000,4,6.5,10.000000,05eb27de70c8c7270b1f4a7172f5487b8ad0326b36fe28...,Villarreal
3076,68b5d6ae026ac303d72e0129,Mikautadze,delantero,18,51b890f5b986415a2c000012,6.285714,5.142857,5.714286,29203734,14,5,4.000000,10,7.0,9.000000,2f9653eee45de91ca6f1fc66af6942930a59fbd3a00f13...,Villarreal
2584,68b5d6ae026ac303d72e0129,Mikautadze,delantero,17,51b890f5b986415a2c000012,6.285714,5.500000,5.923077,27140144,13,5,5.714286,9,9.5,4.000000,9623ca8e3fc3773b7028a41c856a24b39a5510edac57ca...,Villarreal


In [26]:
final_df.shape

(7363, 17)

### ASK ABOUT HOME AVERAGE AND AWAY AVERAGE

How to handle it

In [27]:
# # Replace entire table
# final_df.to_sql('player_points', engine, if_exists='replace', index=False)
# logging.info("Saved player_points to database")

In [28]:
df_yamal=final_df[final_df['name']=='Lamine Yamal'].sort_values(by='round', ascending=False)

In [29]:
df_yamal[['round', 'target_points']]

,round,target_points
9175,26,NaN
8642,25,6.0
8109,24,-1.0
7576,23,12.0
5014,22,14.0
4484,21,14.0
3961,20,12.0
3448,19,5.0
2927,18,12.0
2415,17,7.0


In [30]:
# Connect to database
engine = create_engine(
    f"postgresql+psycopg2://{db_config['username']}@{db_config['host']}:"
    f"{db_config['port']}/{db_config['database']}")

# Save player points
final_df.to_sql('player_points_def', engine, if_exists='replace', index=False)
logging.info("Saved player_points to database")

In [31]:
# Load historical match data
df_la_liga = pd.read_sql("SELECT * FROM la_liga_matches", engine)
df_liga_next = pd.read_csv('/Users/rodrigo/football-data-analytics/futmondo_points_predict/data/la_liga_next_rounds copy.csv')

# Standardize team names
df_la_liga = standardize_team_names(df_la_liga, is_historical=True)
df_liga_next = standardize_team_names(df_liga_next, is_historical=False)

# Check for team name mismatches
historical_teams = set(df_la_liga['HomeTeam'].unique()) | set(df_la_liga['AwayTeam'].unique())
upcoming_teams = set(df_liga_next['Home Team'].unique()) | set(df_liga_next['Away Team'].unique())
missing_teams = upcoming_teams - historical_teams
if missing_teams:
    logging.warning(f"Teams in upcoming matches not found in historical data: {missing_teams}")

# Predict upcoming matches
team_stats = get_team_stats(df_la_liga)
predictions_df = predict_upcoming_matches(df_liga_next, df_la_liga, team_stats)

# Combine historical and predicted data
updated_df = pd.concat([df_la_liga, predictions_df], ignore_index=True)

# Add matchup probabilities and create advanced features
df_with_matchups = add_matchup_probabilities(final_df, updated_df)
df_enriched = create_advanced_features(df_with_matchups)
df_enriched = clean_numpy_values(df_enriched)

# Save enriched training data
# df_enriched.to_sql('full_training_data', engine, if_exists='replace', index=False)
# logging.info("Saved full_training_data to database")
# logging.info("Database update completed successfully")

/var/folders/r5/bx2jhb6n64n4zm91gw712nr00000gn/T/ipykernel_19554/3744637843.py:21: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  updated_df = pd.concat([df_la_liga, predictions_df], ignore_index=True)
/Users/rodrigo/football-data-analytics/futmondo_points_predict/src/utils.py:94: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  ).fillna(0.0)
/Users/rodrigo/football-data-analytics/futmondo_points_predict/src/utils.py:101: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a

In [32]:
df_la_liga.head(2)

,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,AvgH,AvgD,AvgA,Prob_Home,Prob_Draw,Prob_Away,Total,Prob_Home_Norm,Prob_Draw_Norm,Prob_Away_Norm,Round
0,15/08/2025,Girona,Vallecano,1.0,3.0,A,2.24,3.26,3.26,0.446429,0.306748,0.306748,1.059926,0.421189,0.289406,0.289406,1
1,15/08/2025,Villarreal,Oviedo,2.0,0.0,H,1.38,4.74,8.11,0.724638,0.210970,0.123305,1.058913,0.684322,0.199233,0.116445,1


In [41]:
df_with_matchups.columns

Index(['player_id', 'name', 'role', 'round', 'team_id', 'home_average',
       'away_average', 'overall_average', 'current_price', 'matches_played',
       'rating', 'match_minus_1', 'match_minus_2', 'last_2_average',
       'target_points', 'unique_id', 'team', 'matchup_prob_win',
       'matchup_prob_draw', 'matchup_prob_loss', 'is_home', 'opponent'],
      dtype='object')

In [33]:
df_mika=df_enriched[df_enriched['name']=='Mikautadze'].sort_values(by='round', ascending=False)

In [34]:
df_mika[['round', 'target_points']]

,round,target_points
9421,26,NaN
8888,25,3.000000
8355,24,12.000000
7822,23,9.000000
5159,22,15.000000
4630,21,3.000000
4106,20,3.000000
3598,19,10.000000
3076,18,9.000000
2584,17,4.000000


In [35]:
df_enriched.shape

(7363, 39)

In [36]:
df_enriched.head()

,player_id,name,role,round,team_id,home_average,away_average,overall_average,current_price,matches_played,...,location_adjusted_average,matchup_strength,team_expected_performance,delantero_matchup_bonus,centrocampista_matchup_bonus,defensa_matchup_bonus,portero_matchup_bonus,home_matchup_boost,difficult_matchup,easy_matchup
0,52013ee178b20d7f07000351,Josan,centrocampista,13,51b889b1e401a15f2c0000f0,4.400000,1.500000,3.571429,1000000,7,...,4.400000,-0.570438,0.555189,0.0,0.150752,0.000000,0.000000,0.062813,1,0
1,521a999b5865e5687000001c,David Soria,portero,13,504e581e4d8bec9a670000cd,7.571429,3.750000,5.533333,20975958,15,...,7.571429,-0.355844,0.828432,0.0,0.000000,0.000000,0.689820,0.092138,1,0
2,5203972e2e80d2950b00016f,J. Musso,portero,13,504e581e4d8bec9a670000c8,0.000000,4.000000,4.000000,1000000,1,...,4.000000,0.355844,1.895964,0.0,0.000000,0.000000,1.223586,0.000000,0,1
3,57363a66ad212396073bce9f,Zubeldia,defensa,13,504e581e4d8bec9a670000ce,4.000000,3.666667,3.833333,5422151,12,...,3.666667,0.008957,1.360839,0.0,0.000000,0.504479,0.000000,0.000000,0,0
4,525d4f895231f4d645000795,Iván Villar,portero,13,504e581e4d8bec9a670000d9,0.000000,0.000000,0.000000,1000000,0,...,0.000000,-0.047644,1.273327,0.0,0.000000,0.000000,0.947077,0.000000,0,0


In [47]:
df_enriched.to_sql('full_training_data', engine, if_exists='replace', index=False)

659